In [1]:
import os
import pandas as pd
from typing import List, Dict

In [ ]:
def input_ensemble_results(res_dir: str, filter: List[str] = [], anti_filter: List[str] = []) -> None:
    csv_files = [file for file in os.listdir(res_dir) 
                 if file.endswith(".csv") and
                 all(f.lower() in file.lower() for f in filter) and
                 not any(f.lower() in file.lower() for f in anti_filter)
                ]
    
    pd_files = []    
    
    for csv_file in csv_files:
        file_dir = os.path.join(res_dir, csv_file)
        pd_files.append((csv_file, pd.read_csv(file_dir)))
    
    sample_file = pd_files[0][1]
    task_ids = sample_file['task_id'].to_list()

    res_df = pd.DataFrame()
    res_df["task_id"] = task_ids
    
    for pd_file in pd_files:
        file_name = pd_file[0]
        file_data = pd_file[1]
        res_df[file_name.split(".csv")[0]] = file_data.loc[:, "failure_type"]

    for task_id in task_ids:
        assertion_count = 0
        pass_count = 0
        for pd_file in pd_files:
            file_data = pd_file[1]
            row = file_data.loc[file_data["task_id"] == task_id, "failure_type"]
            value = row.iloc[0]
            if not isinstance(value, float) and AssertionError.__name__ not in value:
                continue
            elif isinstance(value, float):
                pass_count += 1
            else:
                assertion_count += 1

        if pass_count == 0 and assertion_count == 0:
            res_df.loc[res_df['task_id'] == task_id, 'ensemble_results'] = "Invalid_Ensemble_Outcome"
        elif pass_count >= assertion_count:
            res_df.loc[res_df['task_id'] == task_id, 'ensemble_results'] = float('nan')
        else:
            res_df.loc[res_df['task_id'] == task_id, 'ensemble_results'] = AssertionError.__name__

    res_df.to_csv("test.csv")


In [3]:
input_ensemble_results(
    "/Users/jin/Downloads/Test_results_26:09:2025/code_generation/gpt-5", 
    filter=['HumanEval', ], 
    anti_filter=["no_mutation"])

/var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/ipykernel_53184/1325267872.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Ensemble_Outcome' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  res_df.loc[res_df['task_id'] == task_id, 'ensemble_results'] = "Invalid_Ensemble_Outcome"
